## Task 2

### End to End Family MSA 

In [ ]:
import os
import sys
import io
import csv
import json
import re
import math
import time
import shutil
import platform
import subprocess
import warnings
from collections import Counter, defaultdict
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

from Bio import SeqIO, AlignIO, Phylo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

try:
    import logomaker as lm
    HAS_LOGOMAKER = True
except ImportError:
    HAS_LOGOMAKER = False
    print("logomaker not found. Logo plots will not be generated.")

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

def _ensure_pip_packages(packages):
    missing = []
    for pkg, import_name in packages:
        try:
            __import__(import_name)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing missing packages: {', '.join(missing)}")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", *missing]
        )


In [ ]:
PROJECT_ROOT = Path("") 
BIN_DIR      = PROJECT_ROOT / "bin"
DATA_DIR     = PROJECT_ROOT / "data"
OUT_DIR      = PROJECT_ROOT / "download"
FIG_DIR      = OUT_DIR / "figures"

In [ ]:
for d in (BIN_DIR, DATA_DIR, OUT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
_IS_WINDOWS = platform.system() == "Windows"
_IS_LINUX   = platform.system() == "Linux"

In [ ]:
MUSCLE_URLS = {
    "Linux":   "https://github.com/rcedgar/muscle/releases/download/v5.1/muscle5.1.linux_intel64",
    "Windows": "https://github.com/rcedgar/muscle/releases/download/v5.1/muscle5.1.win64.exe",
}

CLUSTALO_URL = "https://conda.anaconda.org/bioconda/linux-64/clustalo-1.2.4-hdbdd923_8.tar.bz2"
ARGTABLE2_URL = "https://conda.anaconda.org/conda-forge/linux-64/argtable2-2.13-h14c3975_1001.tar.bz2"



In [ ]:
def _download(url, dest, label):
    if dest.exists() and dest.stat().st_size > 1000:
        print(f"  {label}: already present ({dest.stat().st_size:,} bytes)")
        return
    print(f"  {label}: downloading from {url}")
    dest.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(url, stream=True, timeout=120,
                     headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    with open(dest, "wb") as fh:
        for chunk in r.iter_content(chunk_size=1 << 16):
            fh.write(chunk)
    print(f"  {label}: saved -> {dest} ({dest.stat().st_size:,} bytes)")


def _extract_bz2(tarball_path, dest_dir, label):
    import tarfile
    print(f"  {label}: extracting {tarball_path} -> {dest_dir}")
    with tarfile.open(tarball_path, "r:bz2") as tf:
        tf.extractall(dest_dir)


def bootstrap_muscle():
    os_name = platform.system()
    url = MUSCLE_URLS.get(os_name)
    if not url:
        raise RuntimeError(f"No MUSCLE URL for OS: {os_name}")
    if _IS_WINDOWS:
        muscle_path = BIN_DIR / "muscle.exe"
    else:
        muscle_path = BIN_DIR / "muscle"
    if not (muscle_path.exists() and muscle_path.stat().st_size > 1000):
        _download(url, muscle_path, "MUSCLE 5.1")
        if not _IS_WINDOWS:
            muscle_path.chmod(0o755)

    try:
        if _IS_WINDOWS:
            res = subprocess.run([str(muscle_path), "--version"],
                                  capture_output=True, text=True, timeout=30)
        else:
            res = subprocess.run([str(muscle_path), "--version"],
                                  capture_output=True, text=True, timeout=30)
        if res.returncode == 0:
            ver = (res.stdout or res.stderr).strip().splitlines()[0]
            print(f"  MUSCLE verified: {ver}")
        else:
            print(f"  WARNING: MUSCLE verification returned rc={res.returncode}")
    except Exception as e:
        print(f"  WARNING: MUSCLE verification failed: {e}")
    return str(muscle_path)


def bootstrap_clustal_omega():
    if not _IS_LINUX:
        print("  Clustal Omega: not bootstrapped on this OS")
        return None
    clustalo_path = BIN_DIR / "clustalo"
    libs_dir     = BIN_DIR / "libs"
    libs_dir.mkdir(parents=True, exist_ok=True)

    if not (clustalo_path.exists() and clustalo_path.stat().st_size > 1000):
        tarball = BIN_DIR / "clustalo.tar.bz2"
        _download(CLUSTALO_URL, tarball, "Clustal Omega (bioconda)")
        _extract_bz2(tarball, BIN_DIR / "_clustalo_extract", "Clustal Omega")
        extracted_bin = BIN_DIR / "_clustalo_extract" / "bin" / "clustalo"
        shutil.copy(extracted_bin, clustalo_path)
        clustalo_path.chmod(0o755)

    argtable_lib = libs_dir / "libargtable2.so.0"
    if not argtable_lib.exists():
        tarball = BIN_DIR / "argtable2.tar.bz2"
        _download(ARGTABLE2_URL, tarball, "argtable2 (conda-forge)")
        _extract_bz2(tarball, BIN_DIR / "_argtable2_extract", "argtable2")
        extracted_lib = BIN_DIR / "_argtable2_extract" / "lib" / "libargtable2.so.0"
        shutil.copy(extracted_lib, argtable_lib)

    os.environ["LD_LIBRARY_PATH"] = (
        str(libs_dir) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    )
    # Verification
    try:
        res = subprocess.run([str(clustalo_path), "--version"],
                             capture_output=True, text=True, timeout=30,
                             env=os.environ)
        if res.returncode == 0:
            ver = (res.stdout or res.stderr).strip().splitlines()[0]
            print(f"  Clustal Omega verified: {ver}")
        else:
            print(f"  WARNING: Clustal Omega verification returned rc={res.returncode}")
    except Exception as e:
        print(f"  WARNING: Clustal Omega verification failed: {e}")
    return str(clustalo_path)

MUSCLE_BIN = bootstrap_muscle()
CLUSTALO_BIN = bootstrap_clustal_omega()

MAFFT_BIN    = str(BIN_DIR / "mafft" / "mafft.bat")

  MUSCLE verified: muscle 5.1.linux64 [12f0e2]
  Clustal Omega verified: 1.2.4


In [ ]:
PROTEIN_QUERY = "TP53"
FAMILY_QUERY  = 'family:"P53 family" AND reviewed:true'
MAX_SEQUENCES = 40  

UNIPROT_SEARCH_URL = "https://rest.uniprot.org/uniprotkb/search"

EBI_CLUSTALO_RUN    = "https://www.ebi.ac.uk/Tools/services/rest/clustalo/run/"
EBI_CLUSTALO_STATUS = "https://www.ebi.ac.uk/Tools/services/rest/clustalo/status/{job_id}"
EBI_CLUSTALO_RESULT = "https://www.ebi.ac.uk/Tools/services/rest/clustalo/result/{job_id}/{fmt}"
EBI_EMAIL = "you@example.com" 

In [ ]:
IS_WINDOWS = platform.system() == "Windows"
IS_LINUX   = platform.system() == "Linux"
IS_MAC     = platform.system() == "Darwin"


def _find_local_clustalo():
    if CLUSTALO_BIN and Path(CLUSTALO_BIN).is_file():
        return CLUSTALO_BIN
    p = shutil.which("clustalo") or shutil.which("clustalo-wrapper")
    if p:
        return p
    candidates = [
        BIN_DIR / "clustalo",
        BIN_DIR / "clustalo-wrapper",
        Path("/usr/bin/clustalo"),
        Path("/usr/local/bin/clustalo"),
        Path.home() / "bin" / "clustalo",
        Path(os.environ.get("CONDA_PREFIX", "")) / "Scripts" / "clustalo.exe",
        Path(os.environ.get("USERPROFILE", "")) / "bin" / "clustalo.exe",
    ]
    for c in candidates:
        try:
            if c.is_file():
                return str(c)
        except Exception:
            pass
    return None


LOCAL_CLUSTALO = _find_local_clustalo()


def _run_local_clustal_omega(binary_path, input_fasta, output_aln):
    """Run a local clustalo binary."""
    cmd = [binary_path, "-i", input_fasta, "-o", output_aln,
           "--force", "--outfmt=fasta"]
    print(f"  CMD: {' '.join(cmd)}")
    env = os.environ.copy()
    if IS_LINUX or IS_MAC:
        env["LD_LIBRARY_PATH"] = (
            str(BIN_DIR / "libs") + ":"
            + env.get("LD_LIBRARY_PATH", "")
        )
    res = subprocess.run(cmd, capture_output=True, text=True, timeout=300, env=env)
    if res.returncode != 0:
        raise RuntimeError(
            f"Local Clustal Omega failed (rc={res.returncode})\n{res.stderr[:500]}"
        )
    aln = AlignIO.read(output_aln, "fasta")
    print(f"  Local Clustal Omega: {len(aln)} seqs x {aln.get_alignment_length()} cols")
    return aln


def _run_ebi_clustal_omega(input_fasta, output_aln, email=EBI_EMAIL,
                            outfmt="fa", poll_interval=10, max_wait=900,
                            verbose=True):
    if verbose:
        print(f"  EBI Clustal Omega: submitting {Path(input_fasta).name}")
        print(f"  (email={email}; expect ~1-3 min queue time on the free tier)")

    with open(input_fasta) as fh:
        fasta_text = fh.read()
    resp = requests.post(
        EBI_CLUSTALO_RUN,
        data={"email": email, "sequence": fasta_text, "outfmt": outfmt},
        headers={"Accept": "text/plain"},
        timeout=60,
    )
    resp.raise_for_status()
    job_id = resp.text.strip()
    if not job_id or "error" in job_id.lower():
        raise RuntimeError(f"EBI job submission failed: {resp.text[:500]}")
    if verbose:
        print(f"  EBI job ID: {job_id}")
        print(f"  Polling every {poll_interval}s (max {max_wait}s)...")

    waited = 0
    while waited < max_wait:
        s = requests.get(
            EBI_CLUSTALO_STATUS.format(job_id=job_id), timeout=30
        ).text.strip()
        if s == "FINISHED":
            if verbose:
                print(f"  Job finished after {waited}s.")
            break
        if s in ("ERROR", "FAILURE", "NOT_FOUND"):
            raise RuntimeError(f"EBI job {job_id} failed with status: {s}")
        time.sleep(poll_interval)
        waited += poll_interval
        if verbose and waited % (poll_interval * 6) == 0:
            print(f"    ...{waited}s elapsed, status={s}")
    else:
        raise TimeoutError(
            f"EBI job {job_id} did not finish within {max_wait}s"
        )

    fmt_map = {"fa": "fasta", "clustal": "clustal",
               "phylip": "phylip", "stockholm": "stockholm"}
    r = requests.get(
        EBI_CLUSTALO_RESULT.format(job_id=job_id, fmt=outfmt), timeout=120
    )
    r.raise_for_status()
    aligned_text = r.text

    with open(output_aln, "w") as fh:
        fh.write(aligned_text)
    if verbose:
        print(f"  Saved alignment -> {output_aln}")

    aln = AlignIO.read(io.StringIO(aligned_text), fmt_map.get(outfmt, "fasta"))
    if verbose:
        print(f"  EBI Clustal Omega: {len(aln)} seqs x {aln.get_alignment_length()} cols")
    return aln


def run_clustal_omega(input_fasta, output_aln, email=EBI_EMAIL):
    if LOCAL_CLUSTALO:
        print(f"  Using local Clustal Omega binary: {LOCAL_CLUSTALO}")
        try:
            return _run_local_clustal_omega(
                LOCAL_CLUSTALO, str(input_fasta), str(output_aln)
            )
        except Exception as e:
            print(f"  Local binary failed: {e}")
            print(f"  Falling back to EBI REST API...")
    else:
        print(f"  Using the EBI REST API (free, but expect a queue).")
    return _run_ebi_clustal_omega(str(input_fasta), str(output_aln), email)



In [ ]:
def search_uniprot(query, max_seqs):
    print(f"Searching UniProt for: {query}")
    # FASTA payload
    r = requests.get(UNIPROT_SEARCH_URL,
                     params={"query": query, "format": "fasta", "size": max_seqs},
                     timeout=60,
                     headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    fasta_text = r.text
    records = list(SeqIO.parse(io.StringIO(fasta_text), "fasta"))
    print(f"  Retrieved {len(records)} sequences.")

    # JSON metadata payload
    r2 = requests.get(UNIPROT_SEARCH_URL,
                      params={"query": query, "format": "json", "size": max_seqs},
                      timeout=60,
                      headers={"User-Agent": "Mozilla/5.0"})
    r2.raise_for_status()
    json_data = r2.json()

    rows = []
    for entry in json_data.get("results", []):
        desc_obj = entry.get("proteinDescription", {})
        rec_name = desc_obj.get("recommendedName", {}).get("fullName", {}).get("value", "")
        if not rec_name:
            alt = desc_obj.get("submissionNames", [])
            if alt:
                rec_name = alt[0].get("fullName", {}).get("value", "")
        genes = entry.get("genes", [])
        gene_name = genes[0].get("geneName", {}).get("value", "") if genes else ""
        rows.append({
            "accession": entry.get("primaryAccession", ""),
            "uniprot_id": entry.get("uniProtkbId", ""),
            "gene": gene_name,
            "protein_name": rec_name,
            "organism": entry.get("organism", {}).get("scientificName", ""),
            "length": entry.get("sequence", {}).get("length", 0),
        })
    return records, fasta_text, pd.DataFrame(rows)


In [ ]:
records, fasta_text, meta_df = search_uniprot(FAMILY_QUERY, MAX_SEQUENCES)

Searching UniProt for: family:"P53 family" AND reviewed:true
  Retrieved 40 sequences.


In [ ]:
fasta_path = DATA_DIR / "p53_family.fasta"
with open(fasta_path, "w") as fh:
    SeqIO.write(records, fh, "fasta")

# Save the metadata CSV
meta_path = OUT_DIR / "uniprot_metadata.csv"
meta_df.to_csv(meta_path, index=False)

meta_df.head()

,accession,uniprot_id,gene,protein_name,organism,length
0,O15350,P73_HUMAN,TP73,Tumor protein p73,Homo sapiens,636
1,P04637,P53_HUMAN,TP53,Cellular tumor antigen p53,Homo sapiens,393
2,Q9H3D4,P63_HUMAN,TP63,Tumor protein 63,Homo sapiens,680
3,P10361,P53_RAT,Tp53,Cellular tumor antigen p53,Rattus norvegicus,391
4,Q9JJP6,P63_RAT,Tp63,Tumor protein 63,Rattus norvegicus,680


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.hist(meta_df["length"], bins=20, color="#4A6FA5", edgecolor="black")
ax.axvline(meta_df["length"].mean(), color="red", linestyle="--",
           label=f"Mean = {meta_df['length'].mean():.1f} aa")
ax.set_xlabel("Sequence length (amino acids)")
ax.set_ylabel("Count")
ax.set_title(f"Sequence length distribution — {len(meta_df)} p53-family members")
ax.legend()
fig.savefig(str(FIG_DIR / "length_distribution.png"), dpi=150)
plt.show()

In [ ]:
def run_muscle(input_fasta, output_aln):
    if not os.path.exists(MUSCLE_BIN):
        raise FileNotFoundError(
            f"MUSCLE binary not found at {MUSCLE_BIN}. "
        )
    cmd = [MUSCLE_BIN, "-align", input_fasta, "-output", output_aln]
    print(f"  CMD: {' '.join(cmd)}")
    res = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    if res.returncode != 0:
        raise RuntimeError(f"MUSCLE failed (rc={res.returncode})\n{res.stderr[:500]}")
    aln = AlignIO.read(output_aln, "fasta")
    print(f"  MUSCLE: {len(aln)} seqs x {aln.get_alignment_length()} cols")
    return aln


def compare_alignments(clustal_aln, muscle_aln):
    def sp_score(aln):
        n = len(aln)
        score = 0
        for i in range(aln.get_alignment_length()):
            col = [str(rec.seq[i]) for rec in aln[:n]]
            for a in range(n):
                for b in range(a + 1, n):
                    if col[a] != "-" and col[b] != "-" and col[a] == col[b]:
                        score += 1
        return score
    clustal_sp = sp_score(clustal_aln)
    muscle_sp  = sp_score(muscle_aln)
    print(f"  Columns  ClustalOmega={clustal_aln.get_alignment_length()}  "
          f"MUSCLE={muscle_aln.get_alignment_length()}")
    print(f"  SP score ClustalOmega={clustal_sp:,}  MUSCLE={muscle_sp:,}")
    return {
        "clustal_columns": clustal_aln.get_alignment_length(),
        "muscle_columns":  muscle_aln.get_alignment_length(),
        "clustal_sp_score": clustal_sp,
        "muscle_sp_score":  muscle_sp,
    }


In [ ]:
clustal_aln_path = OUT_DIR / "alignment_clustal_omega.fasta"
muscle_aln_path  = OUT_DIR / "alignment_muscle.fasta"

clustal_aln = run_clustal_omega(str(fasta_path), str(clustal_aln_path))

muscle_aln  = run_muscle(str(fasta_path), str(muscle_aln_path))

cmp = compare_alignments(clustal_aln, muscle_aln)

  Using local Clustal Omega binary: bin/clustalo
  CMD: bin/clustalo -i data/p53_family.fasta -o download/alignment_clustal_omega.fasta --force --outfmt=fasta


  Local Clustal Omega: 40 seqs x 950 cols
  CMD: bin/muscle -align data/p53_family.fasta -output download/alignment_muscle.fasta
  MUSCLE: 40 seqs x 1049 cols
  Columns  ClustalOmega=950  MUSCLE=1049
  SP score ClustalOmega=155,467  MUSCLE=159,509


In [ ]:
AlignIO.write(clustal_aln, str(OUT_DIR / "alignment_clustal_omega.phy"), "phylip-relaxed")
AlignIO.write(muscle_aln,  str(OUT_DIR / "alignment_muscle.phy"),       "phylip-relaxed")

primary_aln = clustal_aln


In [ ]:
def shannon_entropy(column):
    residues = [r for r in column if r not in ("-", "X")]
    if not residues:
        return 0.0
    counts = Counter(residues)
    n = len(residues)
    return -sum((c / n) * math.log2(c / n) for c in counts.values())

In [ ]:
def conservation_analysis(aln):
    n_seqs = len(aln)
    n_cols = aln.get_alignment_length()
    entropy  = np.zeros(n_cols)
    consensus_identity = np.zeros(n_cols)
    gap_freq = np.zeros(n_cols)
    consensus = []
    for i in range(n_cols):
        col = [str(rec.seq[i]) for rec in aln]
        entropy[i] = shannon_entropy(col)
        non_gap = [r for r in col if r != "-"]
        if not non_gap:
            consensus.append("-")
            consensus_identity[i] = 0
            gap_freq[i] = 1.0
        else:
            c = Counter(non_gap)
            top_res, top_count = c.most_common(1)[0]
            consensus.append(top_res)
            consensus_identity[i] = top_count / len(non_gap)
            gap_freq[i] = col.count("-") / len(col)
    return {
        "entropy": entropy,
        "consensus": "".join(consensus),
        "consensus_identity": consensus_identity,
        "gap_freq": gap_freq,
        "n_cols": n_cols,
        "n_seqs": n_seqs,
    }


In [ ]:
cons = conservation_analysis(primary_aln)

In [ ]:
cons_path = OUT_DIR / "consensus_sequence.txt"
with open(cons_path, "w") as fh:
    fh.write(f">p53_family_consensus (clustal_omega)\n{cons['consensus']}\n")

In [ ]:
def find_conserved_regions(consensus, consensus_identity, min_len=5, threshold=0.9):
    regions = []
    start = None
    for i, ci in enumerate(consensus_identity):
        if ci >= threshold and consensus[i] != "-":
            if start is None:
                start = i
        else:
            if start is not None and (i - start) >= min_len:
                regions.append((start, i, consensus[start:i]))
            start = None
    if start is not None and (len(consensus_identity) - start) >= min_len:
        regions.append((start, len(consensus_identity),
                        consensus[start:len(consensus_identity)]))
    return regions


In [ ]:
regions = find_conserved_regions(cons["consensus"], cons["consensus_identity"])

In [ ]:
for s, e, motif in regions:
    print(f"  cols {s + 1:>4}-{e:<4}  motif: {motif}")

regions_path = OUT_DIR / "conserved_regions.tsv"
with open(regions_path, "w") as fh:
    fh.write("start_col\tend_col\tmotif\n")
    for s, e, motif in regions:
        fh.write(f"{s + 1}\t{e}\t{motif}\n")

  cols    7-11    motif: DSQLS
  cols   86-113   motif: ITHLSLMSTHPSESGRQPELIKTRTRTP
  cols  121-126   motif: LQQTSR
  cols  208-220   motif: SYENDGSRELSET
  cols  453-459   motif: HLIRVEG
  cols  507-512   motif: MCNSSC
  cols  514-522   motif: GGMNRRPIL
  cols  547-554   motif: CACPGRDR


In [ ]:
PROSITE_PATTERNS = {
    "PS00001 N-glycosylation site (N-{P}-[ST]-{P})":
        re.compile(r"N[^P][ST][^P]"),
    "PS00004 Protein kinase C (PKC) phosphorylation ([ST]-x-[RK])":
        re.compile(r"[ST][^P][RK]"),
    "PS00005 Casein kinase II phosphorylation ([ST]-x(2)-[DE])":
        re.compile(r"[ST][^P]{2}[DE]"),
    "PS00006 Tyrosine kinase phosphorylation ([RK]-x(2,3)-[DE]-x(2,3)-Y)":
        re.compile(r"[RK].{2,3}[DE].{2,3}Y"),
    "PS00007 cAMP/cGMP-dependent kinase ([RK](2)-x-[ST])":
        re.compile(r"[RK]{2}.[ST]"),
    "PS00008 N-myristoylation site (G-{EDRKHPFYW}-x(2)-[STAGCN]-{P})":
        re.compile(r"G[^EDRKHPFYW]{2}[STAGCN][^P]"),
    "PS01042 P53 DNA-binding domain signature (L-F-E-[EA]-[STA])":
        re.compile(r"LFE[EA][STA]"),
    "PS50053 P53 tetramerisation motif ([STA]-x(2)-[STA]-x(3)-[STA])":
        re.compile(r"[STA].{2}[STA].{3}[STA]"),
}

In [ ]:
def scan_prosite_motifs(records):
    summary = defaultdict(lambda: {"n_hits": 0, "seqs": set()})
    hits = defaultdict(list)
    for rec in records:
        seq = str(rec.seq)
        for motif_name, pattern in PROSITE_PATTERNS.items():
            for m in pattern.finditer(seq):
                hits[motif_name].append({
                    "seq_id": rec.id,
                    "start": m.start() + 1,
                    "end":   m.end(),
                    "match": m.group(),
                })
                summary[motif_name]["n_hits"] += 1
                summary[motif_name]["seqs"].add(rec.id)
    rows = []
    for motif_name in PROSITE_PATTERNS:
        rows.append({
            "motif": motif_name,
            "n_hits": summary[motif_name]["n_hits"],
            "n_seqs_with_hit": len(summary[motif_name]["seqs"]),
            "fraction_seqs": len(summary[motif_name]["seqs"]) / len(records) if records else 0,
        })
    return pd.DataFrame(rows), hits
motif_df, motif_hits = scan_prosite_motifs(records)
motif_df


,motif,n_hits,n_seqs_with_hit,fraction_seqs
0,PS00001 N-glycosylation site (N-{P}-[ST]-{P}),88,40,1.000
1,PS00004 Protein kinase C (PKC) phosphorylation...,254,40,1.000
2,PS00005 Casein kinase II phosphorylation ([ST]...,217,40,1.000
3,PS00006 Tyrosine kinase phosphorylation ([RK]-...,40,31,0.775
4,PS00007 cAMP/cGMP-dependent kinase ([RK](2)-x-...,18,13,0.325
5,PS00008 N-myristoylation site (G-{EDRKHPFYW}-x...,117,40,1.000
6,PS01042 P53 DNA-binding domain signature (L-F-...,0,0,0.000
7,PS50053 P53 tetramerisation motif ([STA]-x(2)-...,202,40,1.000


In [ ]:
motif_path = OUT_DIR / "prosite_motifs_summary.csv"
motif_df.to_csv(motif_path, index=False)

hits_path = OUT_DIR / "prosite_motif_hits.json"
with open(hits_path, "w") as fh:
    json.dump({k: v for k, v in motif_hits.items()},
              fh, indent=2, default=str)

In [ ]:
def build_phylo_tree(aln, out_newick, out_plot):
    calculator = DistanceCalculator("identity")
    dm = calculator.get_distance(aln)
    constructor = DistanceTreeConstructor()
    tree = constructor.nj(dm)
    tree.ladderize()

    Phylo.write(tree, out_newick, "newick")

    Phylo.draw_ascii(tree, file=sys.stdout)

    fig, ax = plt.subplots(figsize=(10, max(8, len(aln) * 0.25)),
                           constrained_layout=True)
    Phylo.draw(tree, axes=ax, do_show=False,
               branch_labels=None,
               label_func=lambda c: c.name,
               show_confidence=False)
    ax.set_xlabel("Branch length (p-distance)")
    ax.set_title("Neighbor-Joining Phylogenetic Tree — p53 Family",
                 fontsize=13, fontweight="bold")
    fig.savefig(out_plot, dpi=150, bbox_inches="tight")
    plt.show()
    return tree

tree_newick = OUT_DIR / "p53_tree.newick"
tree_plot   = FIG_DIR / "phylogenetic_tree.png"
tree = build_phylo_tree(primary_aln, str(tree_newick), str(tree_plot))


  __ sp|Q9WUR6|P53_CAVPO
 |
 |_ sp|O36006|P53_MARMO
 |
 |_ sp|Q95330|P53_RABIT
 |
 | _ sp|Q9TTA1|P53_TUPBE
 ||
 ||, sp|P04637|P53_HUMAN
 |||
 | , sp|P13481|P53_CHLAE
 | |
 | , sp|P56423|P53_MACFA
 | |
_| , sp|P61260|P53_MACFU
 | |
 | | sp|P56424|P53_MACMU
 |
 |   , sp|Q00366|P53_MESAU
 |___|
 |   | sp|O09185|P53_CRIGR
 |
 |  _ sp|P02340|P53_MOUSE
 |_|
 | |_ sp|P10361|P53_RAT
 |
 | _ sp|Q29537|P53_CANLF
 ,|
 ||_ sp|P41685|P53_FELCA
 |
 | _ sp|Q9TUB2|P53_PIG
 ||
 || sp|Q8SPZ3|P53_DELLE
 ||
 ||, sp|P51664|P53_SHEEP
 |||
 | |, sp|P67938|P53_BOSIN
 | ||
 |  | sp|P67939|P53_BOVIN
 |
 |    __ sp|Q64662|P53_OTOBE
 |  _|
 | | |   ____ sp|Q29480|P53_EQUAS
 | | |__|
 | |    |_ sp|P79892|P53_HORSE
 | |
 | |  _______ sp|P10360|P53_CHICK
 |_| |
   | |  ______ sp|P07193|P53_XENLA
   | |,|
   | |||   _____ sp|O57538|P53_XIPHE
   | |||__|
   | ||   | _____ sp|O12946|P53_PLAFE
   |_||   ||
     ||    |_____ sp|P79820|P53_ORYLA
     ||
     ||  ______ sp|P25035|P53_ONCMY
     ||_|
     || |_____ sp|O9337

In [ ]:
def plot_conservation(cons, out_path):
    x = np.arange(cons["n_cols"])
    fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True,
                             constrained_layout=True)
    ax = axes[0]
    ax.plot(x, cons["entropy"], color="#4A6FA5", linewidth=0.8)
    ax.fill_between(x, cons["entropy"], 0, color="#4A6FA5", alpha=0.2)
    ax.axhline(0.5, color="orange", linestyle="--", linewidth=0.7,
               label="High-conservation threshold (0.5 bits)")
    ax.set_ylabel("Shannon entropy (bits)")
    ax.set_title("Per-column conservation profile — p53 family MSA",
                 fontsize=12, fontweight="bold")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(alpha=0.3)

    ax = axes[1]
    ax.plot(x, cons["consensus_identity"] * 100, color="#2E7D32", linewidth=0.8)
    ax.fill_between(x, cons["consensus_identity"] * 100, 0,
                    color="#2E7D32", alpha=0.2)
    ax.set_xlabel("Alignment column")
    ax.set_ylabel("Consensus identity (%)")
    ax.set_ylim(0, 105)
    ax.grid(alpha=0.3)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
plot_conservation(cons, str(FIG_DIR / "conservation_plot.png"))

In [ ]:
%pip install logomaker
import logomaker as lm

Note: you may need to restart the kernel to use updated packages.


In [ ]:
def plot_sequence_logo(aln, out_path, window=60):
    if not HAS_LOGOMAKER:
        print("logomaker not available")
        return
    n_cols = aln.get_alignment_length()
    cons_vec = []
    for i in range(n_cols):
        col = Counter(str(rec.seq[i]) for rec in aln)
        n = sum(v for k, v in col.items() if k != "-")
        top = max((v for k, v in col.items() if k != "-"), default=0)
        cons_vec.append(top / n if n else 0)
    if n_cols > window:
        best, best_score = 0, -1
        for s in range(0, n_cols - window, 5):
            sc = np.mean(cons_vec[s:s + window])
            if sc > best_score:
                best_score, best = sc, s
        start, end = best, best + window
    else:
        start, end = 0, n_cols
    counts = defaultdict(lambda: np.zeros(end - start))
    for rec in aln:
        for i, r in enumerate(str(rec.seq)[start:end]):
            if r not in ("-", "X"):
                counts[r][i] += 1
    df = pd.DataFrame(counts).fillna(0)

    fig, ax = plt.subplots(figsize=(14, 4), constrained_layout=True)
    lm.Logo(df, ax=ax, color_scheme="chemistry")
    ax.set_xlabel("Alignment column")
    ax.set_ylabel("Residue count")
    ax.set_title(f"Sequence logo — p53 family alignment (cols {start + 1}-{end})",
                 fontsize=12, fontweight="bold")
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()

plot_sequence_logo(primary_aln, str(FIG_DIR / "sequence_logo.png"))

In [ ]:
def plot_motif_summary(motif_df, out_path):
    if motif_df.empty:
        return
    fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
    short = [m.split(" ", 2)[0] + " " + (m.split(" ", 2)[1] if len(m.split()) > 1 else "")[:25]
             for m in motif_df["motif"]]
    y = motif_df["n_seqs_with_hit"].values
    x = np.arange(len(y))
    ax.bar(x, y, color="#8B4A6B", edgecolor="black")
    ax.set_xticks(x)
    ax.set_xticklabels(short, rotation=35, ha="right", fontsize=8)
    ax.set_ylabel("Number of sequences with hit")
    ax.set_title("PROSITE motif distribution across p53 family",
                 fontsize=12, fontweight="bold")
    for i, v in enumerate(y):
        ax.text(i, v + 0.2, str(v), ha="center", fontsize=8)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()

plot_motif_summary(motif_df, str(FIG_DIR / "prosite_motif_summary.png"))


In [ ]:
summary = {
    "protein_query": PROTEIN_QUERY,
    "family_query":  FAMILY_QUERY,
    "n_sequences":   len(records),
    "max_sequences": MAX_SEQUENCES,
    "clustal_omega": {
        "alignment_path": str(clustal_aln_path),
        "n_columns":      clustal_aln.get_alignment_length(),
        "sp_score":       cmp["clustal_sp_score"],
    },
    "muscle": {
        "alignment_path": str(muscle_aln_path),
        "n_columns":       muscle_aln.get_alignment_length(),
        "sp_score":        cmp["muscle_sp_score"],
    },
    "conservation": {
        "mean_entropy_bits":         float(cons["entropy"].mean()),
        "strictly_conserved_cols":   int((cons["entropy"] == 0).sum()),
        "highly_conserved_cols":     int((cons["entropy"] < 0.5).sum()),
        "mean_consensus_identity_pct": float(cons["consensus_identity"].mean() * 100),
        "mean_gap_freq_pct":         float(cons["gap_freq"].mean() * 100),
        "consensus_path": str(cons_path),
        "regions_path":    str(regions_path),
        "n_conserved_regions": len(regions),
    },
    "motifs": {
        "summary_csv":      str(motif_path),
        "hits_json":        str(hits_path),
        "n_motifs_scanned": len(PROSITE_PATTERNS),
        "top_motif": motif_df.sort_values("n_seqs_with_hit", ascending=False)
                              .iloc[0].to_dict() if not motif_df.empty else None,
    },
    "phylogeny": {
        "newick_path": str(tree_newick),
        "plot_path":   str(tree_plot),
    },
    "figures": {
        "conservation_plot": str(FIG_DIR / "conservation_plot.png"),
        "sequence_logo":     str(FIG_DIR / "sequence_logo.png"),
        "motif_summary":     str(FIG_DIR / "prosite_motif_summary.png"),
        "phylogenetic_tree": str(tree_plot),
        "length_distribution": str(FIG_DIR / "length_distribution.png"),
    },
    "metadata_csv": str(OUT_DIR / "uniprot_metadata.csv"),
}
summary_path = OUT_DIR / "pipeline_summary.json"
with open(summary_path, "w") as fh:
    json.dump(summary, fh, indent=2)